In [3]:
import gc
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet34_Weights, resnet34

In [2]:
DATASET_ROOT = Path("/content/datasets")
IMAGE_ROOT = DATASET_ROOT / "images"
MASK_ROOT = DATASET_ROOT / "crack-seg-semantic" / "masks"

MANIFEST_PATH = Path(
    "/content/"
    "vision_unit_02_outputs/"
    "block_02/"
    "semantic_mask_manifest.csv"
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_03"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 416
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_AMP = (DEVICE.type == "cuda")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("AMP:", USE_AMP)
print("Output:", OUTPUT_ROOT)

PyTorch: 2.11.0+cpu
Device: cpu
AMP: False
Output: /content/drive/MyDrive/vision_unit_02_outputs/block_03


**Reproducibility**

In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

**Load only train and validation rows**

In [ ]:
required_paths = [
    IMAGE_ROOT / "train",
    IMAGE_ROOT / "val",
    MASK_ROOT / "train",
    MASK_ROOT / "val",
    MANIFEST_PATH,
]

for path in required_paths:
    assert path.exists(), (
        f"Missing: {path}"
    )

manifest = pd.read_csv(MANIFEST_PATH)

working_manifest = (
    manifest[manifest["split"].isin(["train", "val"])]
    .copy()
    .reset_index(drop=True)
)

train_frame = (
    working_manifest[working_manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_frame = (
    working_manifest[working_manifest["split"] == "val"]
    .copy()
    .reset_index(drop=True)
)


print("Train:", len(train_frame))
print("Validation:", len(val_frame))
print("Test rows used:",(working_manifest["split"] == "test").sum())


Train: 3717
Validation: 200
Test rows used: 0
